In [14]:
# [1] 필요한 라이브러리 불러오기 및 .env 파일에서 API Key 로딩

from dotenv import load_dotenv
import os

# .env 파일에서 환경변수 불러오기
load_dotenv()

# KAMIS-like API key 변수명
API_KEY = os.getenv("KAMIS_API_KEY")

# 확인용 출력 (주의: 실제 배포 시에는 노출 금지)
print("API Key Loaded:", "✅" if API_KEY else "❌ 오류: API Key 없음")



API Key Loaded: ✅


In [15]:
# [11] 전체 품목 중 '배추', '쌀', '양파', '상추', '사과' 포함된 중분류 이름/코드 자동 추출

import requests
import xml.etree.ElementTree as ET

def auto_find_middle_codes(api_key, keywords, chunk_size=1000, max_rows=13249):
    base_url = "http://211.237.50.150:7080/openapi"
    grid_id = "Grid_20141221000000000120_1"
    
    found = {}
    
    for start in range(1, max_rows + 1, chunk_size):
        end = min(start + chunk_size - 1, max_rows)
        url = f"{base_url}/{api_key}/xml/{grid_id}/{start}/{end}"

        try:
            response = requests.get(url)
            response.raise_for_status()
            root = ET.fromstring(response.text)
            rows = root.findall(".//row")
            
            for row in rows:
                name = row.findtext("PRDLST_NM")
                code = row.findtext("PRDLST_CD")
                for keyword in keywords:
                    if keyword in name and keyword not in found:
                        found[keyword] = {"품목명": name, "코드": code}
                        print(f"✅ {keyword} → {name} (코드: {code})")
                if len(found) == len(keywords):
                    break
        except Exception as e:
            print(f"❌ 오류: {e}")
            break
        
        if len(found) == len(keywords):
            break

    return found

# ✅ 실행
keywords = ["쌀", "배추", "양파", "상추", "사과"]
code_dict = auto_find_middle_codes(API_KEY, keywords)

print("\n📌 최종 중분류 코드 매핑 결과:")
for k, v in code_dict.items():
    print(f"{k}: {v['품목명']} (코드: {v['코드']})")


✅ 사과 → 사과 (코드: 19I9)
✅ 양파 → 양파 (코드: 1201)
✅ 배추 → 양배추 (코드: 1004)
✅ 상추 → 상추 (코드: 1005)
✅ 쌀 → 쌀 (코드: 0103)

📌 최종 중분류 코드 매핑 결과:
사과: 사과 (코드: 19I9)
양파: 양파 (코드: 1201)
배추: 양배추 (코드: 1004)
상추: 상추 (코드: 1005)
쌀: 쌀 (코드: 0103)


In [16]:
# [13] 소분류(SPCIES_NM) 중 '배추' 포함된 항목 찾기

def find_species_names_with(keyword: str, api_key: str, start=1, end=13249):
    url = f"http://211.237.50.150:7080/openapi/{api_key}/xml/Grid_20141221000000000120_1/{start}/{end}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        root = ET.fromstring(response.text)

        rows = root.findall(".//row")

        print(f"📋 소분류 품종명(SPCIES_NM) 중 '{keyword}' 포함된 항목:\n")
        count = 0
        for row in rows:
            species = row.find("SPCIES_NM")
            code = row.find("SPCIES_CD")
            prd = row.find("PRDLST_NM")
            if species is not None and keyword in species.text:
                print(f"- {species.text} (품목: {prd.text}, 소분류코드: {code.text})")
                count += 1
        print(f"\n총 {count}개 발견됨")
    except Exception as e:
        print("❌ 오류:", e)

# ✅ 실행
find_species_names_with("", API_KEY)


📋 소분류 품종명(SPCIES_NM) 중 '' 포함된 항목:


총 0개 발견됨


In [17]:
# [1] 수집 설정 셀: 기간, 품목 코드 매핑, 날짜 생성

import os
from datetime import datetime, timedelta
import pandas as pd

# ✅ API 키 로딩
API_KEY = os.getenv("KAMIS_API_KEY")  # .env에서 자동 불러옴

# ✅ 수집할 중분류 품목 코드 매핑 (배추 제외)
ITEM_CODES = {
    "쌀": "0103",
    "양파": "1201",
    "상추": "1005",
    "사과": "19I9"
}

# ✅ 수집할 날짜 리스트 생성: 2015-01-01 ~ 2024-12-01, 매월 1일
def generate_monthly_dates(start_year=2015, end_year=2024):
    date_list = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            date_str = f"{year}{month:02}01"
            date_list.append(date_str)
    return date_list

DATES = generate_monthly_dates()

# ✅ 검증 출력
print(f"📅 수집 날짜 수: {len(DATES)}개")
print(f"🎯 수집 품목: {list(ITEM_CODES.keys())}")
print(f"🔑 API 키: {'OK' if API_KEY else '❌ 없음'}")


📅 수집 날짜 수: 120개
🎯 수집 품목: ['쌀', '양파', '상추', '사과']
🔑 API 키: OK


In [18]:
# [2-1] 전체 품목 코드 중 '쌀', '양파', '상추', '사과'가 들어간 MID 추출

import requests
import xml.etree.ElementTree as ET

def find_mid_codes_by_keyword(api_key, keywords, chunk_size=1000, max_rows=13249):
    base_url = "http://211.237.50.150:7080/openapi"
    grid_id = "Grid_20141221000000000120_1"
    
    found = {}

    for start in range(1, max_rows + 1, chunk_size):
        end = min(start + chunk_size - 1, max_rows)
        url = f"{base_url}/{api_key}/xml/{grid_id}/{start}/{end}"

        try:
            response = requests.get(url)
            response.raise_for_status()
            root = ET.fromstring(response.text)
            rows = root.findall(".//row")

            for row in rows:
                mid = row.findtext("PRDLST_CD")
                mid_name = row.findtext("PRDLST_NM")

                if mid and mid_name:
                    for keyword in keywords:
                        if keyword in mid_name and keyword not in found:
                            found[keyword] = {"MID": mid, "MIDNAME": mid_name}
                            print(f"✅ {keyword} → {mid_name} (코드: {mid})")

                if len(found) == len(keywords):
                    break

        except Exception as e:
            print(f"❌ 오류: {e}")
            break

        if len(found) == len(keywords):
            break

    return found

# ✅ 실행: 목표 품목에 대한 MID 코드 찾기
keywords = ["쌀", "양파", "상추", "사과"]
mid_code_map = find_mid_codes_by_keyword(API_KEY, keywords)

print("\n📌 최종 MID 코드 매핑 결과:")
for k, v in mid_code_map.items():
    print(f"{k}: {v['MIDNAME']} (MID: {v['MID']})")


✅ 사과 → 사과 (코드: 19I9)
✅ 양파 → 양파 (코드: 1201)
✅ 상추 → 상추 (코드: 1005)
✅ 쌀 → 쌀 (코드: 0103)

📌 최종 MID 코드 매핑 결과:
사과: 사과 (MID: 19I9)
양파: 양파 (MID: 1201)
상추: 상추 (MID: 1005)
쌀: 쌀 (MID: 0103)


In [19]:
# [3] 실시간 경락 데이터 수집 함수 (MID 기반, WHSALCD 고정)

def fetch_wholesale_data(item_name, mid_code, saledate, whsalcd="110001", start=1, end=1000, api_key=API_KEY):
    """
    - item_name: '쌀', '양파' 등
    - mid_code: 중분류 품목 코드
    - saledate: 'YYYYMMDD' 형식
    - whsalcd: 도매시장 코드 (기본값: 서울가락 110001)
    """
    base_url = "http://211.237.50.150:7080/openapi"
    grid_id = "Grid_20240625000000000654_1"
    url = f"{base_url}/{api_key}/xml/{grid_id}/{start}/{end}"

    params = {
        "SALEDATE": saledate,
        "WHSALCD": whsalcd,
        "MID": mid_code,
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        root = ET.fromstring(response.text)
        rows = root.findall(".//row")
        
        results = []
        for row in rows:
            result = {
                "날짜": saledate,
                "품목명": item_name,
                "도매시장": row.findtext("WHSALNAME"),
                "법인명": row.findtext("CMPNAME"),
                "산지": row.findtext("SANNAME"),
                "도매단가": row.findtext("COST"),
                "거래량": row.findtext("QTY"),
                "규격": row.findtext("STD"),
                "입찰시각": row.findtext("SBIDTIME")
            }
            results.append(result)
        return results
    
    except Exception as e:
        print(f"❌ 요청 오류: {saledate} / {item_name} →", e)
        return []


In [20]:
# ✅ 다른 품목: 사과 / 가을 시즌
test = fetch_wholesale_data("사과", "19I9", "20211001")
print(f"📦 데이터 수: {len(test)}")
if test:
    print("예시:", test[0])


📦 데이터 수: 0


In [21]:
# [4] 도매시장 실시간 경락 API에서 사용된 MID 목록 추출

import requests
import xml.etree.ElementTree as ET

def get_mid_codes_used(saledate="20240502", whsalcd="110001", api_key=API_KEY, start=1, end=1000):
    """
    도매시장 경락 API에서 실제로 존재하는 MID + MIDNAME 코드 목록 추출
    """
    base_url = "http://211.237.50.150:7080/openapi"
    grid_id = "Grid_20240625000000000654_1"
    url = f"{base_url}/{api_key}/xml/{grid_id}/{start}/{end}"
    params = {
        "SALEDATE": saledate,
        "WHSALCD": whsalcd
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        root = ET.fromstring(response.text)
        rows = root.findall(".//row")

        mid_set = {}
        for row in rows:
            mid = row.findtext("MID")
            midname = row.findtext("MIDNAME")
            if mid and midname:
                mid_set[mid] = midname
        
        print(f"📦 총 MID 코드 개수: {len(mid_set)}개")
        return mid_set

    except Exception as e:
        print("❌ MID 목록 요청 오류:", e)
        return {}

# ✅ 테스트 실행: 2024-05-02 기준
mid_dict = get_mid_codes_used("20240615")
for code, name in mid_dict.items():
    print(f"{code}: {name}")


📦 총 MID 코드 개수: 15개
09: 마늘
08: 홍고추
01: 오이
02: 호박
26: 파프리카
05: 강낭콩
99: 기타
04: 표고버섯
03: 가지
06: 방울토마토
11: 새송이
19: 고구마순
25: 고추잎
10: 부추
31: 청경채


In [22]:
# [5] 도매시장 실시간 API에서 사용된 모든 MID 코드 + 이름 자동 수집

import requests
import xml.etree.ElementTree as ET

def collect_all_mid_codes(api_key=API_KEY, saledate="20240502", whsalcd="110001", max_pages=10, rows_per_page=1000):
    """
    도매시장 실시간 API에서 MID + MIDNAME 전체 목록 수집
    - max_pages: 총 요청할 페이지 수 (1000건 x N)
    """
    base_url = "http://211.237.50.150:7080/openapi"
    grid_id = "Grid_20240625000000000654_1"
    
    mids = {}

    for i in range(max_pages):
        start = i * rows_per_page + 1
        end = start + rows_per_page - 1
        url = f"{base_url}/{api_key}/xml/{grid_id}/{start}/{end}"
        params = {
            "SALEDATE": saledate,
            "WHSALCD": whsalcd
        }

        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            root = ET.fromstring(response.text)
            rows = root.findall(".//row")
            
            if not rows:
                break  # 더 이상 데이터가 없으면 종료
            
            for row in rows:
                mid = row.findtext("MID")
                midname = row.findtext("MIDNAME")
                if mid and midname:
                    mids[mid] = midname

        except Exception as e:
            print(f"❌ 오류: {start}~{end} 요청 실패 → {e}")
            break

    print(f"📦 전체 MID 코드 수: {len(mids)}개")
    return mids

# ✅ 실행
all_mid_dict = collect_all_mid_codes()
for code, name in all_mid_dict.items():
    print(f"{code}: {name}")


📦 전체 MID 코드 수: 39개
26: 호박잎
02: 배
05: 풋고추
01: 사과
03: 포도
04: 딸기
08: 시금치
06: 방울토마토
41: 가죽나물
20: 방풍나물
25: 비타민
24: 새싹
99: 기타
07: 케일
27: 식용허브
09: 마늘
16: 겨자
17: 민들레
10: 부추
11: 비트(붉은사탕무우)
15: 아스파라가스
12: 바나나
31: 청경채
14: 근대
30: 쑥
19: 자몽
13: 파인애플
18: 오렌지
33: 안스리움
21: 비름
39: 무순
40: 잎당귀
28: 로메인
62: 겨자잎
50: 겨자채
52: 고수
34: 아보카도
58: 용과
36: 망고


In [23]:
# [6] 전국 도매시장 순회하며 MID + MIDNAME 탐색 (품목 키워드 포함)

import requests
import xml.etree.ElementTree as ET

# ✅ 도매시장 코드 목록 직접 정의 (총 5개 예시 / 필요시 확장 가능)
whsalcd_list = {
    "110001": "서울가락",
    "110008": "서울강서",
    "210001": "부산엄궁",
    "210005": "부산국제수산",
    "210009": "부산반여",
    "220001": "대구북부"
}

# ✅ 품목 키워드 (MIDNAME에 이 문자열이 포함되어 있는지 확인)
keywords = ["양파", "쌀", "상추", "배추", "사과"]

def find_mids_in_all_markets(api_key, keywords, whsalcd_dict, saledate="20240502"):
    found = {}

    for whsalcd, market_name in whsalcd_dict.items():
        print(f"\n🏢 도매시장: {market_name} ({whsalcd})")
        url = f"http://211.237.50.150:7080/openapi/{api_key}/xml/Grid_20240625000000000654_1/1/1000"
        params = {
            "SALEDATE": saledate,
            "WHSALCD": whsalcd
        }

        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            root = ET.fromstring(response.text)
            rows = root.findall(".//row")

            for row in rows:
                mid = row.findtext("MID")
                midname = row.findtext("MIDNAME")

                if mid and midname:
                    for keyword in keywords:
                        if keyword in midname and keyword not in found:
                            found[keyword] = {
                                "도매시장": market_name,
                                "WHSALCD": whsalcd,
                                "MID": mid,
                                "MIDNAME": midname
                            }
                            print(f"✅ {keyword} 발견 → {market_name}, MID: {mid}, MIDNAME: {midname}")

            if len(found) == len(keywords):
                break  # 모든 키워드를 다 찾았으면 종료

        except Exception as e:
            print(f"❌ 오류 발생 ({market_name}):", e)

    return found

# ✅ 실행
matched_mids = find_mids_in_all_markets(API_KEY, keywords, whsalcd_list)

# ✅ 최종 요약 출력
print("\n📌 최종 MID 매핑 결과:")
for k, v in matched_mids.items():
    print(f"{k}: {v['MIDNAME']} (MID: {v['MID']}, 도매시장: {v['도매시장']})")



🏢 도매시장: 서울가락 (110001)
✅ 양파 발견 → 서울가락, MID: 01, MIDNAME: 양파
✅ 배추 발견 → 서울가락, MID: 04, MIDNAME: 양배추

🏢 도매시장: 서울강서 (110008)
✅ 사과 발견 → 서울강서, MID: 01, MIDNAME: 사과

🏢 도매시장: 부산엄궁 (210001)
✅ 상추 발견 → 부산엄궁, MID: 05, MIDNAME: 상추

🏢 도매시장: 부산국제수산 (210005)

🏢 도매시장: 부산반여 (210009)

🏢 도매시장: 대구북부 (220001)

📌 최종 MID 매핑 결과:
양파: 양파 (MID: 01, 도매시장: 서울가락)
배추: 양배추 (MID: 04, 도매시장: 서울가락)
사과: 사과 (MID: 01, 도매시장: 서울강서)
상추: 상추 (MID: 05, 도매시장: 부산엄궁)


In [24]:
# [7-1] 월별 경락 데이터 수집 + 정리 + CSV 저장 함수

import pandas as pd
import os
from datetime import datetime

def collect_monthly_price_data(item_name, mid_code, whsalcd, market_name, api_key=API_KEY,
                                start_year=2015, end_year=2024, save_folder="output"):
    """
    지정된 품목명, MID, 도매시장코드 기준으로 2015~2024 월별 데이터를 수집하고 CSV로 저장
    """
    results = []

    # 월별 1일 날짜 생성
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            saledate = f"{year}{month:02}01"

            # 도매시장 실시간 경락 정보 요청
            data = fetch_wholesale_data(item_name, mid_code, saledate, whsalcd)
            if not data:
                continue

            for row in data:
                results.append({
                    "날짜": f"{year}-{month:02}-01",
                    "품목명": item_name,
                    "도매시장": row.get("도매시장"),
                    "단량당 금액": int(row.get("도매단가") or 0),
                    "거래량(kg)": int(float(row.get("거래량") or 0)),
                    "생산지": row.get("산지") or "-"
                })

    # DataFrame 변환 + 평균 단가 계산
    if not results:
        print(f"❌ {item_name} 데이터 없음")
        return

    df = pd.DataFrame(results)
    df_grouped = df.groupby(["날짜", "품목명", "도매시장", "생산지"]).agg({
        "단량당 금액": "mean",
        "거래량(kg)": "sum"
    }).reset_index()

    # 저장 폴더 생성 및 CSV 저장
    os.makedirs(save_folder, exist_ok=True)
    file_path = os.path.join(save_folder, f"{item_name}_{market_name}.csv")
    df_grouped.to_csv(file_path, index=False, encoding="utf-8-sig")
    print(f"✅ 저장 완료: {file_path}")


In [25]:
collect_monthly_price_data("사과", "01", "110008", "서울강서", start_year=2015, end_year=2024)


✅ 저장 완료: output\사과_서울강서.csv
